# Đọc bộ dữ liệu chung — TV2/TV3/TV4

Chạy 3 ô code dưới đây trong Colab CPU mới. Notebook này chỉ xác minh và đọc
các tập đã chốt; không tải lại OpenML, không chia lại dữ liệu, không huấn luyện.
Sửa `BASE` theo thư mục Drive được chia sẻ/sao chép của bạn.
Nếu checksum sai, tải/sao chép lại đầy đủ thư mục; không sửa checksum để bỏ qua lỗi.


In [ ]:
# Cài Java17 nếu phiên Colab chưa có; lỗi cài đặt phải dừng ngay.
import os
import subprocess
from pathlib import Path
java_candidates = sorted(Path("/usr/lib/jvm").glob("java-17-openjdk-*"))
if not java_candidates:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True)
    java_candidates = sorted(Path("/usr/lib/jvm").glob("java-17-openjdk-*"))
assert java_candidates, "Không tìm thấy Java17!"
java17_path = java_candidates[0]
os.environ["JAVA_HOME"] = str(java17_path)
os.environ["PATH"] = str(java17_path / "bin") + ":" + os.environ["PATH"]
subprocess.run([str(java17_path / "bin/java"), "-version"], check=True)


In [ ]:
%pip install -q "pyspark==4.0.1"
import os, sys
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType
os.environ["PYSPARK_PYTHON"] = sys.executable
assert pyspark.__version__ == "4.0.1", "Khởi động lại phiên nếu vừa đổi phiên bản PySpark đã import."
spark = (SparkSession.builder.appName("Airlines_Receive_Data")
         .master("local[2]").config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.session.timeZone", "UTC").getOrCreate())
assert spark.version == "4.0.1"


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json
drive.mount("/content/drive")

# Sửa thành vị trí thư mục được chia sẻ/sao chép trong Drive của BẠN.
BASE = Path("/content/drive/MyDrive/BigData_Topic20/01_data_frozen/tv1_v1")
assert BASE.is_dir(), "Chưa thấy tv1_v1: kiểm tra tài khoản, shortcut và đường dẫn Drive."
manifest = json.loads((BASE / "manifest.json").read_text(encoding="utf-8"))
expected_hashes = json.loads((BASE / "split_checksums.json").read_text(encoding="utf-8"))
assert manifest["source"]["openml_id"] == 1169
assert manifest["counts"] == {"train":376749, "val":81606, "test":81028}
assert set(expected_hashes) == {"train", "val", "test"}

def checksums(folder):
    result = {}
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            h = hashlib.sha256()
            with path.open("rb") as stream:
                for block in iter(lambda:stream.read(1024*1024), b""):
                    h.update(block)
            result[path.relative_to(folder).as_posix()] = h.hexdigest()
    return result

parts = {}
for name in ("train", "val", "test"):
    folder = BASE / "splits" / name
    assert (folder / "_SUCCESS").is_file(), f"Thiếu/chưa hoàn tất tập {name}"
    assert checksums(folder) == expected_hashes[name], f"{name}: bản sao thiếu/thừa/đổi file"
    part = spark.read.parquet(str(folder))
    assert part.schema == StructType.fromJson(manifest["schemas"][name]), f"{name}: sai schema"
    assert part.count() == manifest["counts"][name], f"{name}: sai số dòng"
    parts[name] = part
    print(name, manifest["counts"][name], "dòng — ĐẠT")

train, val, test = parts["train"], parts["val"], parts["test"]
target = "Delay"
feature_columns = ["Airline", "Flight", "AirportFrom", "AirportTo", "DayOfWeek", "Time", "Length"]
# Length_invalid có thể thêm nếu nhóm muốn dùng cờ thiếu; tuyệt đối không thêm row_id/fold/Delay.
assert not {"row_id", "fold", "Delay"}.intersection(feature_columns)
print("Đã sẵn sàng cho TV2 làm đặc trưng và các thành viên huấn luyện.")


## Quy tắc dùng tiếp

- Đọc README_HANDOFF.md trong tv1_v1. Nhãn Delay=1 là trễ; row_id/fold không làm predictor.
- Fit các bước học từ dữ liệu chỉ trên Train. Target encoding Train dùng OOF, gồm cả prior.
- Chọn mô hình/ngưỡng trên Validation; Test để đánh giá cuối, không dùng để chọn đặc trưng.
- Fold trong Train phục vụ OOF theo bản ghi, không bảo đảm tách các lịch bay giống nhau.
- Nếu đánh giá bằng CV bên trong Train: khôi phục Length null ở Length_invalid=1,
  học lại imputer/encoder/scaler trong mỗi phần huấn luyện. Target encoding cần OOF
  bên trong phần huấn luyện đó; không tái dùng một bảng OOF toàn Train cho mọi lượt CV.
- Khoảng77% Val/Test trùng bộ đặc trưng với Train. Báo cáo giới hạn của đánh giá theo
  bản ghi; không coi đây là phép đo khả năng dự báo tương lai/lịch bay hoàn toàn mới.
